# ETL histórico de tasas ponderadas — crédito de consumo

Este notebook construye una base histórica reproducible desde la API de Datos Abiertos de la Superintendencia Financiera de Colombia.

## Objetivo

Extraer el histórico disponible para establecimientos bancarios, crédito de consumo y producto Libre inversión; limpiar nombres de bancos y rangos de monto; calcular tasas ponderadas por número de créditos; y exportar bases listas para análisis y modelado.

## Salidas esperadas

- `outputs/tasas_raw_historico.csv`: datos crudos descargados con tipos básicos normalizados.
- `outputs/tasas_ponderadas_banco_rango_mes.csv`: tasa ponderada por banco, mes y rango de monto.
- `outputs/tasas_ponderadas_banco_mes.csv`: tasa ponderada mensual por banco.
- `outputs/tasas_ponderadas_total_mes.csv`: serie mensual total.
- `outputs/diccionario_bancos.csv`: equivalencias de nombres de bancos.
- `outputs/diccionario_rangos.csv`: orden y normalización de rangos de monto.

**Documentación (Markdown):** [README.md](README.md) · [Informe ETL por sección](docs/INFORME_ETL.md)

## 1. Configuración

Los parámetros de esta celda controlan el alcance del ETL. Por defecto se mantiene la consulta del notebook original: establecimientos bancarios, crédito de consumo y producto Libre inversión. No se excluyen bancos en la descarga; las exclusiones o agrupaciones se dejan como columnas revisables.

In [3]:
from pathlib import Path
from datetime import date
from dateutil.relativedelta import relativedelta
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import requests

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)

URL_API = "https://www.datos.gov.co/resource/w9zh-vetq.json"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

PARAMETROS = {
    "nombre_tipo_entidad": "BC-ESTABLECIMIENTO BANCARIO",
    "tipo_de_credito": "Consumo",
    "producto_de_credito": "Libre inversión",
    "fecha_inicio_manual": None,  # usar date(AAAA, M, D) si se quiere forzar inicio
    "fecha_fin_manual": None,     # usar date(AAAA, M, D) si se quiere forzar fin
    "page_size": 50000,
    "sleep_seconds": 0.25,
    "timeout_seconds": 90,
    "max_retries": 3,
    "retry_backoff_seconds": 3,
    "calcular_conteo_global_diagnostico": False,
    "excluir_rangos_no_informados": True,
}

COLUMNAS_RAW = [
    "nombre_tipo_entidad",
    "nombre_entidad",
    "tipo_de_cr_dito",
    "producto_de_cr_dito",
    "rango_monto_desembolsado",
    "fecha_corte",
    "tasa_efectiva_promedio",
    "numero_de_creditos",
]

print("Configuración cargada")
print(f"Directorio de salida: {OUTPUT_DIR.resolve()}")

Configuración cargada
Directorio de salida: C:\Users\CAMILO\Aprendizage automatico\Juan Camilos 2\outputs


## 2. Funciones base y diagnóstico de la API

Estas funciones centralizan las consultas a Socrata. El diagnóstico busca el rango histórico disponible con los filtros definidos y permite confirmar si la API responde antes de lanzar la extracción completa.

In [6]:
def construir_where_base(parametros: dict, fecha_inicio=None, fecha_fin=None) -> str:
    filtros = [
        f"nombre_tipo_entidad = '{parametros['nombre_tipo_entidad']}'",
        f"tipo_de_cr_dito = '{parametros['tipo_de_credito']}'",
        f"producto_de_cr_dito = '{parametros['producto_de_credito']}'",
        "tasa_efectiva_promedio IS NOT NULL",
        "numero_de_creditos IS NOT NULL",
        "rango_monto_desembolsado IS NOT NULL",
    ]

    if fecha_inicio is not None:
        filtros.append(f"fecha_corte >= '{fecha_inicio.strftime('%Y-%m-%dT00:00:00.000')}'")
    if fecha_fin is not None:
        filtros.append(f"fecha_corte <= '{fecha_fin.strftime('%Y-%m-%dT00:00:00.000')}'")

    return " AND ".join(filtros)


def consultar_api(params: dict, session: requests.Session | None = None) -> list[dict]:
    cliente = session or requests.Session()
    ultimo_error = None

    for intento in range(1, PARAMETROS["max_retries"] + 1):
        try:
            respuesta = cliente.get(
                URL_API,
                params=params,
                timeout=PARAMETROS["timeout_seconds"],
            )
            respuesta.raise_for_status()
            data = respuesta.json()

            if isinstance(data, dict) and "error" in data:
                raise RuntimeError(f"Error de Socrata: {data}")

            return data
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as exc:
            ultimo_error = exc
            if intento == PARAMETROS["max_retries"]:
                break
            espera = PARAMETROS["retry_backoff_seconds"] * intento
            print(f"Reintento {intento}/{PARAMETROS['max_retries']} por timeout/conexión. Esperando {espera}s...")
            time.sleep(espera)

    raise ultimo_error


def consultar_fecha_extrema(where_base: str, orden: str) -> date:
    data = consultar_api({
        "$select": "fecha_corte",
        "$where": where_base,
        "$order": f"fecha_corte {orden}",
        "$limit": 1,
    })
    if not data:
        raise ValueError("La API no devolvió fechas para los filtros configurados")
    return pd.to_datetime(data[0]["fecha_corte"]).date()


def consultar_conteo_global(where_base: str) -> int | None:
    if not PARAMETROS["calcular_conteo_global_diagnostico"]:
        return None

    try:
        data = consultar_api({
            "$select": "count(*) AS registros",
            "$where": where_base,
            "$limit": 1,
        })
        return int(data[0]["registros"]) if data else None
    except requests.exceptions.Timeout:
        print("Conteo global omitido: la API tardó demasiado. La extracción mensual calculará los totales reales.")
        return None


def diagnosticar_api() -> dict:
    where_base = construir_where_base(PARAMETROS)

    # Evita agregaciones globales pesadas sobre todo el histórico.
    fecha_min = consultar_fecha_extrema(where_base, "ASC")
    fecha_max = consultar_fecha_extrema(where_base, "DESC")
    registros_api = consultar_conteo_global(where_base)

    if PARAMETROS["fecha_inicio_manual"] is not None:
        fecha_min = PARAMETROS["fecha_inicio_manual"]
    if PARAMETROS["fecha_fin_manual"] is not None:
        fecha_max = PARAMETROS["fecha_fin_manual"]

    diagnostico = {
        "fecha_inicio": fecha_min.replace(day=1),
        "fecha_fin": fecha_max,
        "registros_api": registros_api,
        "where_base": where_base,
        "nota": "Conteo global omitido por defecto para evitar timeouts; se valida en la extracción mensual.",
    }
    return diagnostico


diagnostico = diagnosticar_api()
diagnostico

{'fecha_inicio': datetime.date(2022, 7, 1),
 'fecha_fin': datetime.date(2026, 3, 27),
 'registros_api': None,
 'where_base': "nombre_tipo_entidad = 'BC-ESTABLECIMIENTO BANCARIO' AND tipo_de_cr_dito = 'Consumo' AND producto_de_cr_dito = 'Libre inversión' AND tasa_efectiva_promedio IS NOT NULL AND numero_de_creditos IS NOT NULL AND rango_monto_desembolsado IS NOT NULL",
 'nota': 'Conteo global omitido por defecto para evitar timeouts; se valida en la extracción mensual.'}

In [8]:
def meses_entre(fecha_inicio: date, fecha_fin: date) -> list[date]:
    meses = []
    actual = fecha_inicio.replace(day=1)
    fin = fecha_fin.replace(day=1)

    while actual <= fin:
        meses.append(actual)
        actual += relativedelta(months=1)

    return meses


meses_a_descargar = meses_entre(diagnostico["fecha_inicio"], diagnostico["fecha_fin"])
print(f"Meses a descargar: {len(meses_a_descargar)}")
print(f"Primer mes: {meses_a_descargar[0]:%Y-%m}")
print(f"Último mes : {meses_a_descargar[-1]:%Y-%m}")

Meses a descargar: 45
Primer mes: 2022-07
Último mes : 2026-03


## 3. Extracción histórica

La extracción se hace por mes para evitar respuestas demasiado grandes. Cada mes se pagina con `$limit` y `$offset`, de modo que el proceso siga funcionando aunque un mes tenga más registros que el límite configurado.

In [11]:
def extraer_mes(fecha_mes: date, session: requests.Session) -> pd.DataFrame:
    mes_inicio = fecha_mes.replace(day=1)
    mes_fin = mes_inicio + relativedelta(months=1) - relativedelta(days=1)
    where_mes = construir_where_base(PARAMETROS, mes_inicio, mes_fin)

    registros = []
    offset = 0
    page_size = PARAMETROS["page_size"]

    while True:
        params = {
            "$select": ", ".join(COLUMNAS_RAW),
            "$where": where_mes,
            "$order": "fecha_corte, nombre_entidad, rango_monto_desembolsado",
            "$limit": page_size,
            "$offset": offset,
        }
        pagina = consultar_api(params, session=session)
        registros.extend(pagina)

        if len(pagina) < page_size:
            break

        offset += page_size
        time.sleep(PARAMETROS["sleep_seconds"])

    df_mes = pd.DataFrame(registros)
    if not df_mes.empty:
        df_mes["mes_descarga"] = mes_inicio.strftime("%Y-%m")
    return df_mes


def extraer_historico(meses: list[date]) -> pd.DataFrame:
    dfs = []
    session = requests.Session()

    print("=" * 72)
    print("EXTRACCIÓN HISTÓRICA")
    print("=" * 72)

    for fecha_mes in meses:
        try:
            df_mes = extraer_mes(fecha_mes, session=session)
            dfs.append(df_mes)
            print(f"{fecha_mes:%Y-%m}: {len(df_mes):>8,} registros")
        except Exception as exc:
            print(f"{fecha_mes:%Y-%m}: ERROR -> {exc}")
            raise
        finally:
            time.sleep(PARAMETROS["sleep_seconds"])

    if not dfs:
        raise ValueError("No se descargó ningún dato")

    df = pd.concat(dfs, ignore_index=True)
    print("-" * 72)
    print(f"Total descargado: {len(df):,} registros")
    print("=" * 72)
    return df


raw = extraer_historico(meses_a_descargar)
raw.head()

EXTRACCIÓN HISTÓRICA
2022-07:      968 registros
2022-08:      827 registros
2022-09:    1,086 registros
2022-10:      861 registros
2022-11:      934 registros
2022-12:    1,124 registros
2023-01:      770 registros
2023-02:      822 registros
2023-03:    1,050 registros
2023-04:      809 registros
2023-05:      756 registros
2023-06:      922 registros
2023-07:      689 registros
2023-08:      649 registros
2023-09:   16,933 registros
2023-10:   65,739 registros
2023-11:   65,657 registros
2023-12:   70,220 registros
2024-01:   47,890 registros
2024-02:   52,901 registros
2024-03:   63,274 registros
2024-04:   55,974 registros
2024-05:   72,686 registros
2024-06:   55,122 registros
2024-07:   58,549 registros
2024-08:   77,603 registros
2024-09:   67,345 registros
2024-10:   71,136 registros
2024-11:   86,199 registros
2024-12:   63,262 registros
2025-01:   86,383 registros
2025-02:   79,180 registros
2025-03:   75,528 registros
2025-04:   83,572 registros
2025-05:  102,391 registros

,nombre_tipo_entidad,nombre_entidad,tipo_de_cr_dito,producto_de_cr_dito,rango_monto_desembolsado,fecha_corte,tasa_efectiva_promedio,numero_de_creditos,mes_descarga
0,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,28.94,2,2022-07
1,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,17.99,112,2022-07
2,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,21.75,1,2022-07
3,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,15.93,238,2022-07
4,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,17.5,1,2022-07


### 3.1 Diagnóstico de volumen descargado

El histórico puede tener cambios de granularidad en la fuente. Esta celda resume el volumen por mes y marca saltos fuertes para revisar antes de modelar o comparar periodos.

In [14]:
raw_diagnostico = raw.copy()
raw_diagnostico["fecha_corte"] = pd.to_datetime(raw_diagnostico["fecha_corte"], errors="coerce")
raw_diagnostico["mes"] = raw_diagnostico["fecha_corte"].dt.to_period("M").astype(str)

volumen_mensual = (
    raw_diagnostico.groupby("mes")
    .agg(
        registros=("mes", "size"),
        fechas_corte=("fecha_corte", "nunique"),
        bancos=("nombre_entidad", "nunique"),
        rangos=("rango_monto_desembolsado", "nunique"),
        productos=("producto_de_cr_dito", "nunique"),
        creditos_reportados=("numero_de_creditos", lambda serie: pd.to_numeric(serie, errors="coerce").sum()),
    )
    .reset_index()
)

volumen_mensual["registros_mes_anterior"] = volumen_mensual["registros"].shift(1)
volumen_mensual["factor_vs_mes_anterior"] = (
    volumen_mensual["registros"] / volumen_mensual["registros_mes_anterior"]
).round(2)

saltos_volumen = volumen_mensual[
    (volumen_mensual["factor_vs_mes_anterior"] >= 3)
    | (volumen_mensual["factor_vs_mes_anterior"] <= 0.33)
].copy()

print("Resumen de volumen descargado")
display(volumen_mensual)

print(f"Meses con saltos fuertes de volumen: {len(saltos_volumen)}")
display(saltos_volumen)

if not saltos_volumen.empty:
    meses_revision = saltos_volumen["mes"].tolist()
    detalle_fechas_corte = (
        raw_diagnostico[raw_diagnostico["mes"].isin(meses_revision)]
        .groupby(["mes", "fecha_corte"])
        .size()
        .reset_index(name="registros")
        .sort_values(["mes", "fecha_corte"])
    )
    display(detalle_fechas_corte.head(200))

Resumen de volumen descargado


,mes,registros,fechas_corte,bancos,rangos,productos,creditos_reportados,registros_mes_anterior,factor_vs_mes_anterior
0,2022-07,968,5,22,1,1,314275,NaN,NaN
1,2022-08,827,4,24,1,1,261794,968.0,0.85
2,2022-09,1086,5,24,1,1,341318,827.0,1.31
3,2022-10,861,4,23,1,1,265278,1086.0,0.79
4,2022-11,934,4,24,1,1,258116,861.0,1.08
...,...,...,...,...,...,...,...,...,...
40,2025-11,81051,4,22,10,1,198740,100971.0,0.80
41,2025-12,77139,4,22,10,1,180612,81051.0,0.95
42,2026-01,89105,5,21,10,1,204077,77139.0,1.16
43,2026-02,80130,4,21,10,1,181177,89105.0,0.90


Meses con saltos fuertes de volumen: 2


,mes,registros,fechas_corte,bancos,rangos,productos,creditos_reportados,registros_mes_anterior,factor_vs_mes_anterior
14,2023-09,16933,5,23,11,1,217390,649.0,26.09
15,2023-10,65739,4,23,10,1,288908,16933.0,3.88


,mes,fecha_corte,registros
0,2023-09,2023-09-01,175
1,2023-09,2023-09-08,160
2,2023-09,2023-09-15,170
3,2023-09,2023-09-22,161
4,2023-09,2023-09-29,16267
5,2023-10,2023-10-06,16280
6,2023-10,2023-10-13,18099
7,2023-10,2023-10-20,15303
8,2023-10,2023-10-27,16057


In [16]:
print("Dimensiones raw:", raw.shape)
display(raw.dtypes.to_frame("tipo"))
display(raw.head())

Dimensiones raw: (2235315, 9)


,tipo
nombre_tipo_entidad,object
nombre_entidad,object
tipo_de_cr_dito,object
producto_de_cr_dito,object
rango_monto_desembolsado,object
fecha_corte,object
tasa_efectiva_promedio,object
numero_de_creditos,object
mes_descarga,object


,nombre_tipo_entidad,nombre_entidad,tipo_de_cr_dito,producto_de_cr_dito,rango_monto_desembolsado,fecha_corte,tasa_efectiva_promedio,numero_de_creditos,mes_descarga
0,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,28.94,2,2022-07
1,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,17.99,112,2022-07
2,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,21.75,1,2022-07
3,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,15.93,238,2022-07
4,BC-ESTABLECIMIENTO BANCARIO,AV Villas,Consumo,Libre inversión,N/A,2022-07-01T00:00:00.000,17.5,1,2022-07


## 4. Rangos originales antes de normalizar

Antes de modificar `rango_monto_desembolsado`, se genera el listado completo de valores originales encontrados en la API. Esta tabla sirve como insumo para decidir cómo agrupar, renombrar u ordenar los rangos.

In [19]:
def clave_rango_original(valor) -> str:
    if pd.isna(valor):
        return "<nulo>"
    texto = re.sub(r"\s+", " ", str(valor).strip()).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    texto = re.sub(r"[^a-z0-9]+", " ", texto).strip()
    return texto or "<vacio>"


raw_rangos = raw.copy()
raw_rangos["fecha_corte"] = pd.to_datetime(raw_rangos["fecha_corte"], errors="coerce")
raw_rangos["mes"] = raw_rangos["fecha_corte"].dt.to_period("M").astype(str)
raw_rangos["rango_original"] = raw_rangos["rango_monto_desembolsado"].fillna("<NULO>").astype(str).str.strip()
raw_rangos["clave_rango_original"] = raw_rangos["rango_original"].map(clave_rango_original)
raw_rangos["numero_de_creditos_num"] = pd.to_numeric(raw_rangos["numero_de_creditos"], errors="coerce")
raw_rangos["tasa_efectiva_promedio_num"] = pd.to_numeric(raw_rangos["tasa_efectiva_promedio"], errors="coerce")

listado_rangos_originales = (
    raw_rangos.groupby(["rango_original", "clave_rango_original"], dropna=False)
    .agg(
        registros=("rango_original", "size"),
        primer_mes=("mes", "min"),
        ultimo_mes=("mes", "max"),
        meses=("mes", "nunique"),
        fechas_corte=("fecha_corte", "nunique"),
        bancos=("nombre_entidad", "nunique"),
        creditos_reportados=("numero_de_creditos_num", "sum"),
        tasa_minima=("tasa_efectiva_promedio_num", "min"),
        tasa_maxima=("tasa_efectiva_promedio_num", "max"),
    )
    .reset_index()
    .sort_values(["primer_mes", "registros", "rango_original"], ascending=[True, False, True])
    .reset_index(drop=True)
)

RANGO_PROPUESTO_ALIASES = {
    "hasta 1 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 1 smlmv menor o igual a 2 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 1 smlmv menor o igual a 3 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 2 smlmv menor o igual a 3 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 3 smlmv menor o igual a 6 smlmv": (2, "Mayor a 3 SMLMV y menor o igual a 6 SMLMV"),
    "mayor a 6 smlmv menor o igual a 12 smlmv": (3, "Mayor a 6 SMLMV y menor o igual a 12 SMLMV"),
    "mayor a 12 smlmv menor o igual a 25 smlmv": (4, "Mayor a 12 SMLMV y menor o igual a 25 SMLMV"),
    "mayor a 25 smlmv menor o igual a 50 smlmv": (5, "Mayor a 25 SMLMV y menor o igual a 100 SMLMV"),
    "mayor a 50 smlmv menor o igual a 100 smlmv": (5, "Mayor a 25 SMLMV y menor o igual a 100 SMLMV"),
    "mayor a 100 smlmv menor o igual a 150 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor a 150 smlmv menor o igual a 300 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor a 150 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor 300 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor a 300 smlmv": (6, "Mayor a 100 SMLMV"),
}

plantilla_normalizacion_rangos = listado_rangos_originales.copy()
propuestas_rango = plantilla_normalizacion_rangos["clave_rango_original"].map(RANGO_PROPUESTO_ALIASES)
plantilla_normalizacion_rangos["rango_orden_propuesto"] = propuestas_rango.map(
    lambda item: item[0] if isinstance(item, tuple) else pd.NA
).astype("Int64")
plantilla_normalizacion_rangos["rango_monto_propuesto"] = propuestas_rango.map(
    lambda item: item[1] if isinstance(item, tuple) else pd.NA
)
plantilla_normalizacion_rangos["accion_revision"] = np.where(
    plantilla_normalizacion_rangos["rango_orden_propuesto"].isna(),
    "revisar/no asignado",
    "agrupar",
)

print(f"Rangos originales encontrados: {len(listado_rangos_originales)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120):
    display(listado_rangos_originales)

print("Plantilla para definir la modificación de rangos:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 120):
    display(plantilla_normalizacion_rangos)

Rangos originales encontrados: 11


,rango_original,clave_rango_original,registros,primer_mes,ultimo_mes,meses,fechas_corte,bancos,creditos_reportados,tasa_minima,tasa_maxima
0,N/A,n a,12933,2022-07,2023-09,15,65,24,3136621,0.00,47.09
1,Mayor a 3 SMLMV menor o igual a 6 SMLMV,mayor a 3 smlmv menor o igual a 6 smlmv,486957,2023-09,2026-03,31,131,26,1159575,0.00,43.39
2,Mayor a 1 SMLMV menor o igual a 3 SMLMV,mayor a 1 smlmv menor o igual a 3 smlmv,458377,2023-09,2026-03,31,131,26,1249618,1.94,42.05
3,Mayor a 6 SMLMV menor o igual a 12 SMLMV,mayor a 6 smlmv menor o igual a 12 smlmv,438676,2023-09,2026-03,31,131,26,1001476,0.00,42.05
4,Mayor a 12 SMLMV menor o igual a 25 SMLMV,mayor a 12 smlmv menor o igual a 25 smlmv,319399,2023-09,2026-03,31,131,26,700965,0.00,42.05
5,Hasta 1 SMLMV,hasta 1 smlmv,199484,2023-09,2026-03,31,131,24,521996,0.00,42.05
6,Mayor a 25 SMLMV menor o igual a 50 SMLMV,mayor a 25 smlmv menor o igual a 50 smlmv,196901,2023-09,2026-03,31,131,24,414882,0.00,42.05
7,Mayor a 50 SMLMV menor o igual a 100 SMLMV,mayor a 50 smlmv menor o igual a 100 smlmv,85400,2023-09,2026-03,31,131,23,192849,0.00,42.04
8,Mayor a 100 SMLMV menor o igual a 150 SMLMV,mayor a 100 smlmv menor o igual a 150 smlmv,23932,2023-09,2026-03,31,131,21,53471,0.00,41.91
9,Mayor a 150 SMLMV menor o igual a 300 SMLMV,mayor a 150 smlmv menor o igual a 300 smlmv,10188,2023-09,2026-03,31,131,21,43014,4.03,42.04


Plantilla para definir la modificación de rangos:


,rango_original,clave_rango_original,registros,primer_mes,ultimo_mes,meses,fechas_corte,bancos,creditos_reportados,tasa_minima,tasa_maxima,rango_orden_propuesto,rango_monto_propuesto,accion_revision
0,N/A,n a,12933,2022-07,2023-09,15,65,24,3136621,0.00,47.09,<NA>,<NA>,revisar/no asignado
1,Mayor a 3 SMLMV menor o igual a 6 SMLMV,mayor a 3 smlmv menor o igual a 6 smlmv,486957,2023-09,2026-03,31,131,26,1159575,0.00,43.39,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,agrupar
2,Mayor a 1 SMLMV menor o igual a 3 SMLMV,mayor a 1 smlmv menor o igual a 3 smlmv,458377,2023-09,2026-03,31,131,26,1249618,1.94,42.05,1,Menor o igual a 3 SMLMV,agrupar
3,Mayor a 6 SMLMV menor o igual a 12 SMLMV,mayor a 6 smlmv menor o igual a 12 smlmv,438676,2023-09,2026-03,31,131,26,1001476,0.00,42.05,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,agrupar
4,Mayor a 12 SMLMV menor o igual a 25 SMLMV,mayor a 12 smlmv menor o igual a 25 smlmv,319399,2023-09,2026-03,31,131,26,700965,0.00,42.05,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,agrupar
5,Hasta 1 SMLMV,hasta 1 smlmv,199484,2023-09,2026-03,31,131,24,521996,0.00,42.05,1,Menor o igual a 3 SMLMV,agrupar
6,Mayor a 25 SMLMV menor o igual a 50 SMLMV,mayor a 25 smlmv menor o igual a 50 smlmv,196901,2023-09,2026-03,31,131,24,414882,0.00,42.05,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV,agrupar
7,Mayor a 50 SMLMV menor o igual a 100 SMLMV,mayor a 50 smlmv menor o igual a 100 smlmv,85400,2023-09,2026-03,31,131,23,192849,0.00,42.04,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV,agrupar
8,Mayor a 100 SMLMV menor o igual a 150 SMLMV,mayor a 100 smlmv menor o igual a 150 smlmv,23932,2023-09,2026-03,31,131,21,53471,0.00,41.91,6,Mayor a 100 SMLMV,agrupar
9,Mayor a 150 SMLMV menor o igual a 300 SMLMV,mayor a 150 smlmv menor o igual a 300 smlmv,10188,2023-09,2026-03,31,131,21,43014,4.03,42.04,6,Mayor a 100 SMLMV,agrupar


## 5. Normalización de datos

Esta sección limpia tipos, fechas y textos. Los bancos se normalizan con un diccionario editable y los rangos originales se agrupan en seis categorías SMLMV: `<=3`, `>3 y <=6`, `>6 y <=12`, `>12 y <=25`, `>25 y <=100` y `>100`.

In [21]:
def limpiar_espacios(valor):
    if pd.isna(valor):
        return np.nan
    return re.sub(r"\s+", " ", str(valor).strip())


def clave_texto(valor) -> str:
    if pd.isna(valor):
        return ""
    texto = limpiar_espacios(valor).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return limpiar_espacios(texto)


BANCO_ALIASES = {
    "av villas": "Banco AV Villas",
    "banco av villas": "Banco AV Villas",
    "banco comercial av villas": "Banco AV Villas",
    "bbva colombia": "BBVA Colombia",
    "banco bilbao vizcaya argentaria colombia": "BBVA Colombia",
    "banco bilbao vizcaya argentaria colombia s a": "BBVA Colombia",
    "banco caja social": "Banco Caja Social",
    "banco caja social s a": "Banco Caja Social",
    "banco davivienda": "Banco Davivienda",
    "davivienda": "Banco Davivienda",
    "banco falabella": "Banco Falabella",
    "banco falabella s a": "Banco Falabella",
    "banco popular": "Banco Popular",
    "banco popular s a": "Banco Popular",
    "banco serfinanza": "Banco Serfinanza",
    "banco serfinanza s a": "Banco Serfinanza",
    "banco union": "Banco Unión",
    "banco de bogota": "Banco de Bogotá",
    "banco de bogota s a": "Banco de Bogotá",
    "banco de occidente": "Banco de Occidente",
    "banco de occidente s a": "Banco de Occidente",
    "bancolombia": "Bancolombia",
    "banco colombia": "Bancolombia",
    "bancoomeva": "Bancoomeva",
    "finandina": "Finandina",
    "banco finandina": "Finandina",
    "itau": "Itaú",
    "itau corpbanca colombia": "Itaú",
    "itau corpbanca colombia s a": "Itaú",
    "lulo bank": "Lulo Bank",
    "bancamia": "Bancamía",
    "bancamia s a": "Bancamía",
    "banco mundo mujer": "Banco Mundo Mujer",
    "banco mundo mujer s a": "Banco Mundo Mujer",
    "banco w": "Banco W",
    "banco w s a": "Banco W",
    "mibanco": "MiBanco",
    "mibanco s a": "MiBanco",
    "scotiabank colpatria": "Banco Davibank",
    "scotiabank colpatria s a": "Banco Davibank",
    "banco gnb sudameris": "Banco GNB Sudameris",
    "banagrario": "Banco Agrario",
    "banco agrario": "Banco Agrario",
    "banco agrario de colombia": "Banco Agrario",
    "banco pichincha": "Banco Pichincha",
    "banco pichincha s a": "Banco Pichincha",
    "coopcentral": "Coopcentral",
    "bancien": "Bancien",
    "banco davibank": "Banco Davibank",
    "banco contactar": "Banco Contactar",
    "banco santander": "Banco Santander",
}


def normalizar_banco(nombre):
    clave = clave_texto(nombre)
    if clave in BANCO_ALIASES:
        return BANCO_ALIASES[clave]
    return limpiar_espacios(nombre)


RANGOS_MONTO = [
    (1, "Menor o igual a 3 SMLMV"),
    (2, "Mayor a 3 SMLMV y menor o igual a 6 SMLMV"),
    (3, "Mayor a 6 SMLMV y menor o igual a 12 SMLMV"),
    (4, "Mayor a 12 SMLMV y menor o igual a 25 SMLMV"),
    (5, "Mayor a 25 SMLMV y menor o igual a 100 SMLMV"),
    (6, "Mayor a 100 SMLMV"),
]

RANGO_ALIASES = {
    "hasta 1 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 1 smlmv menor o igual a 2 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 1 smlmv menor o igual a 3 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 2 smlmv menor o igual a 3 smlmv": (1, "Menor o igual a 3 SMLMV"),
    "mayor a 3 smlmv menor o igual a 6 smlmv": (2, "Mayor a 3 SMLMV y menor o igual a 6 SMLMV"),
    "mayor a 6 smlmv menor o igual a 12 smlmv": (3, "Mayor a 6 SMLMV y menor o igual a 12 SMLMV"),
    "mayor a 12 smlmv menor o igual a 25 smlmv": (4, "Mayor a 12 SMLMV y menor o igual a 25 SMLMV"),
    "mayor a 25 smlmv menor o igual a 50 smlmv": (5, "Mayor a 25 SMLMV y menor o igual a 100 SMLMV"),
    "mayor a 50 smlmv menor o igual a 100 smlmv": (5, "Mayor a 25 SMLMV y menor o igual a 100 SMLMV"),
    "mayor a 100 smlmv menor o igual a 150 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor a 150 smlmv menor o igual a 300 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor a 150 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor 300 smlmv": (6, "Mayor a 100 SMLMV"),
    "mayor a 300 smlmv": (6, "Mayor a 100 SMLMV"),
}


RANGOS_NO_INFORMADOS = {"", "n a", "na", "nulo", "none", "sin informacion", "sin info"}


def es_rango_no_informado(rango) -> bool:
    return clave_texto(rango) in RANGOS_NO_INFORMADOS


def normalizar_rango(rango):
    clave = clave_texto(rango)
    if clave in RANGOS_NO_INFORMADOS:
        return (pd.NA, pd.NA)
    return RANGO_ALIASES.get(clave, (pd.NA, limpiar_espacios(rango)))

In [23]:
def preparar_raw(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()

    for columna in [
        "nombre_tipo_entidad",
        "nombre_entidad",
        "tipo_de_cr_dito",
        "producto_de_cr_dito",
        "rango_monto_desembolsado",
    ]:
        df[columna] = df[columna].map(limpiar_espacios)

    df["fecha_corte"] = pd.to_datetime(df["fecha_corte"], errors="coerce")
    df["mes"] = df["fecha_corte"].dt.to_period("M").astype(str)
    df["tasa_efectiva_promedio"] = pd.to_numeric(df["tasa_efectiva_promedio"], errors="coerce")
    df["numero_de_creditos"] = pd.to_numeric(df["numero_de_creditos"], errors="coerce")

    df["banco_original"] = df["nombre_entidad"]
    df["banco"] = df["nombre_entidad"].map(normalizar_banco)

    df["rango_no_informado"] = df["rango_monto_desembolsado"].map(es_rango_no_informado)
    rangos_normalizados = df["rango_monto_desembolsado"].map(normalizar_rango)
    df["rango_orden"] = rangos_normalizados.map(lambda item: item[0]).astype("Int64")
    df["rango_monto"] = rangos_normalizados.map(lambda item: item[1])

    df["valor_ponderado_tasa"] = df["tasa_efectiva_promedio"] * df["numero_de_creditos"]

    columnas_ordenadas = [
        "fecha_corte",
        "mes",
        "nombre_tipo_entidad",
        "tipo_de_cr_dito",
        "producto_de_cr_dito",
        "banco_original",
        "banco",
        "rango_monto_desembolsado",
        "rango_no_informado",
        "rango_orden",
        "rango_monto",
        "tasa_efectiva_promedio",
        "numero_de_creditos",
        "valor_ponderado_tasa",
        "mes_descarga",
    ]
    return df[columnas_ordenadas]


def filtrar_rangos_para_modelado(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    mascara_excluir = df["rango_no_informado"] | df["rango_orden"].isna()
    excluidos = df.loc[mascara_excluir].copy()

    if PARAMETROS["excluir_rangos_no_informados"]:
        curada = df.loc[~mascara_excluir].copy()
    else:
        curada = df.copy()

    return curada, excluidos


raw_limpia_completa = preparar_raw(raw)
raw_limpia, rangos_excluidos_modelado = filtrar_rangos_para_modelado(raw_limpia_completa)

print("Dimensiones raw limpia completa:", raw_limpia_completa.shape)
print("Registros excluidos por rango no informado/no reconocido:", f"{len(rangos_excluidos_modelado):,}")
print("Dimensiones raw curada para modelado:", raw_limpia.shape)

if not rangos_excluidos_modelado.empty:
    resumen_excluidos_rango = (
        rangos_excluidos_modelado.groupby("rango_monto_desembolsado", dropna=False)
        .agg(registros=("rango_monto_desembolsado", "size"), meses=("mes", "nunique"), bancos=("banco", "nunique"))
        .reset_index()
        .sort_values("registros", ascending=False)
    )
    display(resumen_excluidos_rango)

display(raw_limpia.head())

Dimensiones raw limpia completa: (2235315, 15)
Registros excluidos por rango no informado/no reconocido: 12,933
Dimensiones raw curada para modelado: (2222382, 15)


,rango_monto_desembolsado,registros,meses,bancos
0,N/A,12933,15,24


,fecha_corte,mes,nombre_tipo_entidad,tipo_de_cr_dito,producto_de_cr_dito,banco_original,banco,rango_monto_desembolsado,rango_no_informado,rango_orden,rango_monto,tasa_efectiva_promedio,numero_de_creditos,valor_ponderado_tasa,mes_descarga
12933,2023-09-29,2023-09,BC-ESTABLECIMIENTO BANCARIO,Consumo,Libre inversión,AV Villas,Banco AV Villas,Hasta 1 SMLMV,False,1,Menor o igual a 3 SMLMV,42.05,1,42.05,2023-09
12934,2023-09-29,2023-09,BC-ESTABLECIMIENTO BANCARIO,Consumo,Libre inversión,AV Villas,Banco AV Villas,Hasta 1 SMLMV,False,1,Menor o igual a 3 SMLMV,38.00,1,38.00,2023-09
12935,2023-09-29,2023-09,BC-ESTABLECIMIENTO BANCARIO,Consumo,Libre inversión,AV Villas,Banco AV Villas,Mayor a 100 SMLMV menor o igual a 150 SMLMV,False,6,Mayor a 100 SMLMV,20.90,1,20.90,2023-09
12936,2023-09-29,2023-09,BC-ESTABLECIMIENTO BANCARIO,Consumo,Libre inversión,AV Villas,Banco AV Villas,Mayor a 100 SMLMV menor o igual a 150 SMLMV,False,6,Mayor a 100 SMLMV,36.25,1,36.25,2023-09
12937,2023-09-29,2023-09,BC-ESTABLECIMIENTO BANCARIO,Consumo,Libre inversión,AV Villas,Banco AV Villas,Mayor a 100 SMLMV menor o igual a 150 SMLMV,False,6,Mayor a 100 SMLMV,20.55,1,20.55,2023-09


In [25]:
diccionario_bancos = (
    raw_limpia[["banco_original", "banco"]]
    .drop_duplicates()
    .sort_values(["banco", "banco_original"])
    .reset_index(drop=True)
)

diccionario_rangos = (
    raw_limpia[["rango_monto_desembolsado", "rango_orden", "rango_monto"]]
    .drop_duplicates()
    .sort_values(["rango_orden", "rango_monto_desembolsado"], na_position="last")
    .reset_index(drop=True)
)

print(f"Bancos originales: {diccionario_bancos['banco_original'].nunique()}")
print(f"Bancos normalizados: {diccionario_bancos['banco'].nunique()}")
print(f"Rangos originales: {diccionario_rangos['rango_monto_desembolsado'].nunique()}")
print(f"Rangos no reconocidos: {raw_limpia['rango_orden'].isna().sum():,} registros")

display(diccionario_bancos.head(50))
display(diccionario_rangos)

Bancos originales: 26
Bancos normalizados: 25
Rangos originales: 10
Rangos no reconocidos: 0 registros


,banco_original,banco
0,BBVA Colombia,BBVA Colombia
1,Bancien,Bancien
2,AV Villas,Banco AV Villas
3,Banagrario,Banco Agrario
4,Banco Caja Social S.A.,Banco Caja Social
5,Banco Contactar,Banco Contactar
6,Banco Davibank,Banco Davibank
7,Scotiabank Colpatria S.A.,Banco Davibank
8,Banco Davivienda,Banco Davivienda
9,Banco Falabella S.A.,Banco Falabella


,rango_monto_desembolsado,rango_orden,rango_monto
0,Hasta 1 SMLMV,1,Menor o igual a 3 SMLMV
1,Mayor a 1 SMLMV menor o igual a 3 SMLMV,1,Menor o igual a 3 SMLMV
2,Mayor a 3 SMLMV menor o igual a 6 SMLMV,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV
3,Mayor a 6 SMLMV menor o igual a 12 SMLMV,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV
4,Mayor a 12 SMLMV menor o igual a 25 SMLMV,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV
5,Mayor a 25 SMLMV menor o igual a 50 SMLMV,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV
6,Mayor a 50 SMLMV menor o igual a 100 SMLMV,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV
7,Mayor 300 SMLMV,6,Mayor a 100 SMLMV
8,Mayor a 100 SMLMV menor o igual a 150 SMLMV,6,Mayor a 100 SMLMV
9,Mayor a 150 SMLMV menor o igual a 300 SMLMV,6,Mayor a 100 SMLMV


## 6. Cálculo de tasas ponderadas

La tasa ponderada se calcula como:

\[
\text{tasa ponderada} = \frac{\sum(\text{tasa efectiva promedio} \times \text{número de créditos})}{\sum(\text{número de créditos})}
\]

Se generan tres niveles: banco-rango-mes, banco-mes y total mensual.

In [28]:
def agregar_tasa_ponderada(df: pd.DataFrame, columnas_grupo: list[str]) -> pd.DataFrame:
    agregado = (
        df.groupby(columnas_grupo, dropna=False)
        .agg(
            suma_tasa_credito=("valor_ponderado_tasa", "sum"),
            total_creditos=("numero_de_creditos", "sum"),
            registros_fuente=("numero_de_creditos", "size"),
            fecha_minima=("fecha_corte", "min"),
            fecha_maxima=("fecha_corte", "max"),
        )
        .reset_index()
    )
    agregado["tasa_ponderada"] = agregado["suma_tasa_credito"] / agregado["total_creditos"]
    return agregado


base_calculo = raw_limpia.dropna(
    subset=["fecha_corte", "mes", "banco", "tasa_efectiva_promedio", "numero_de_creditos"]
).copy()
base_calculo = base_calculo[base_calculo["numero_de_creditos"] > 0].copy()

ponderada_banco_rango_mes = agregar_tasa_ponderada(
    base_calculo,
    ["mes", "banco", "rango_orden", "rango_monto"],
).sort_values(["mes", "banco", "rango_orden"]).reset_index(drop=True)

ponderada_banco_mes = agregar_tasa_ponderada(
    base_calculo,
    ["mes", "banco"],
).sort_values(["mes", "banco"]).reset_index(drop=True)

ponderada_total_mes = agregar_tasa_ponderada(
    base_calculo,
    ["mes"],
).sort_values("mes").reset_index(drop=True)

print("Banco-rango-mes:", ponderada_banco_rango_mes.shape)
print("Banco-mes      :", ponderada_banco_mes.shape)
print("Total-mes      :", ponderada_total_mes.shape)

display(ponderada_banco_rango_mes.head())
display(ponderada_total_mes.tail())

Banco-rango-mes: (3591, 10)
Banco-mes      : (696, 8)
Total-mes      : (31, 7)


,mes,banco,rango_orden,rango_monto,suma_tasa_credito,total_creditos,registros_fuente,fecha_minima,fecha_maxima,tasa_ponderada
0,2023-09,BBVA Colombia,1,Menor o igual a 3 SMLMV,13163.74,385,189,2023-09-29,2023-09-29,34.191532
1,2023-09,BBVA Colombia,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,19855.53,578,244,2023-09-29,2023-09-29,34.352128
2,2023-09,BBVA Colombia,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,26798.80,805,307,2023-09-29,2023-09-29,33.290435
3,2023-09,BBVA Colombia,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,22025.51,679,278,2023-09-29,2023-09-29,32.438159
4,2023-09,BBVA Colombia,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV,18196.74,616,299,2023-09-29,2023-09-29,29.540162


,mes,suma_tasa_credito,total_creditos,registros_fuente,fecha_minima,fecha_maxima,tasa_ponderada
26,2025-11,4412861.95,198740,81051,2025-11-07,2025-11-28,22.204196
27,2025-12,4063983.36,180612,77139,2025-12-05,2025-12-26,22.501181
28,2026-01,4566624.06,204077,89105,2026-01-02,2026-01-30,22.376966
29,2026-02,4167228.94,181177,80130,2026-02-06,2026-02-27,23.000872
30,2026-03,3809289.60,162078,75123,2026-03-06,2026-03-27,23.502817


## 7. Validaciones del ETL

Estas validaciones no detienen el proceso si encuentran casos para revisar; producen tablas de control para decidir si se ajustan diccionarios, filtros o reglas de exclusión.

In [31]:
validaciones = {
    "registros_raw_limpia_completa": len(raw_limpia_completa),
    "registros_curados_sin_rango_na": len(raw_limpia),
    "registros_excluidos_rango_no_informado_o_no_reconocido": len(rangos_excluidos_modelado),
    "registros_base_calculo": len(base_calculo),
    "fechas_nulas": int(raw_limpia["fecha_corte"].isna().sum()),
    "tasas_nulas": int(raw_limpia["tasa_efectiva_promedio"].isna().sum()),
    "creditos_nulos": int(raw_limpia["numero_de_creditos"].isna().sum()),
    "creditos_no_positivos": int((raw_limpia["numero_de_creditos"] <= 0).sum()),
    "rangos_no_informados_en_raw_completa": int(raw_limpia_completa["rango_no_informado"].sum()),
    "rangos_no_reconocidos_en_curada": int(raw_limpia["rango_orden"].isna().sum()),
    "tasa_minima": float(base_calculo["tasa_efectiva_promedio"].min()),
    "tasa_maxima": float(base_calculo["tasa_efectiva_promedio"].max()),
    "meses": int(base_calculo["mes"].nunique()),
    "bancos": int(base_calculo["banco"].nunique()),
}

resumen_validaciones = pd.DataFrame(
    validaciones.items(),
    columns=["validacion", "valor"],
)

tasas_extremas = base_calculo.query("tasa_efectiva_promedio < 0 or tasa_efectiva_promedio > 100")
rangos_no_reconocidos = (
    raw_limpia.loc[raw_limpia["rango_orden"].isna(), ["rango_monto_desembolsado"]]
    .drop_duplicates()
    .sort_values("rango_monto_desembolsado")
)

cobertura_banco = (
    base_calculo.groupby("banco")
    .agg(
        primer_mes=("mes", "min"),
        ultimo_mes=("mes", "max"),
        meses=("mes", "nunique"),
        registros=("mes", "size"),
        creditos=("numero_de_creditos", "sum"),
    )
    .sort_values(["meses", "banco"], ascending=[False, True])
    .reset_index()
)

cobertura_mensual = (
    base_calculo.groupby("mes")
    .agg(
        bancos=("banco", "nunique"),
        rangos=("rango_monto", "nunique"),
        registros=("mes", "size"),
        creditos=("numero_de_creditos", "sum"),
    )
    .reset_index()
)

display(resumen_validaciones)
print(f"Filas con tasas extremas: {len(tasas_extremas):,}")
display(cobertura_banco.head(30))
display(cobertura_mensual.tail())

if not rangos_no_reconocidos.empty:
    print("Rangos no reconocidos para revisar:")
    display(rangos_no_reconocidos)

,validacion,valor
0,registros_raw_limpia_completa,2235315.00
1,registros_curados_sin_rango_na,2222382.00
2,registros_excluidos_rango_no_informado_o_no_re...,12933.00
3,registros_base_calculo,2222382.00
4,fechas_nulas,0.00
5,tasas_nulas,0.00
6,creditos_nulos,0.00
7,creditos_no_positivos,0.00
8,rangos_no_informados_en_raw_completa,12933.00
9,rangos_no_reconocidos_en_curada,0.00


Filas con tasas extremas: 0


,banco,primer_mes,ultimo_mes,meses,registros,creditos
0,BBVA Colombia,2023-09,2026-03,31,156752,333561
1,Banco AV Villas,2023-09,2026-03,31,49970,80873
2,Banco Agrario,2023-09,2026-03,31,10905,11001
3,Banco Caja Social,2023-09,2026-03,31,138447,196698
4,Banco Davibank,2023-09,2026-03,31,101294,797150
5,Banco Davivienda,2023-09,2026-03,31,314964,772509
6,Banco Falabella,2023-09,2026-03,31,198912,534597
7,Banco GNB Sudameris,2023-09,2026-03,31,282,311
8,Banco Mundo Mujer,2023-09,2026-03,31,125958,209524
9,Banco Popular,2023-09,2026-03,31,1543,1938


,mes,bancos,rangos,registros,creditos
26,2025-11,22,6,81051,198740
27,2025-12,22,6,77139,180612
28,2026-01,21,6,89105,204077
29,2026-02,21,6,80130,181177
30,2026-03,22,6,75123,162078


## 8. Auditoría pre-modelado

Antes de entrenar modelos, se valida si el dataset sirve para predecir la tasa de interés **dos meses adelante** por `banco + rango_monto`. Esta sección construye el objetivo `t+2`, revisa cobertura temporal por serie y marca qué combinaciones tienen suficiente historia para modelado.

In [34]:
HORIZONTE_PREDICCION_MESES = 2
MIN_MESES_SERIE_MODELO = 18
MIN_OBSERVACIONES_TARGET = 12
MIN_CREDITOS_SERIE = 100


def preparar_dataset_modelo_banco_rango(df: pd.DataFrame, horizonte: int) -> pd.DataFrame:
    dataset = df.copy()
    dataset["mes_periodo"] = pd.PeriodIndex(dataset["mes"], freq="M")
    dataset = dataset.sort_values(["banco", "rango_orden", "mes_periodo"]).reset_index(drop=True)

    grupo = dataset.groupby(["banco", "rango_orden", "rango_monto"], dropna=False)
    dataset[f"tasa_objetivo_{horizonte}m"] = grupo["tasa_ponderada"].shift(-horizonte)
    dataset[f"mes_objetivo_{horizonte}m"] = dataset["mes_periodo"] + horizonte
    dataset[f"creditos_objetivo_{horizonte}m"] = grupo["total_creditos"].shift(-horizonte)

    dataset["mes"] = dataset["mes_periodo"].astype(str)
    dataset[f"mes_objetivo_{horizonte}m"] = dataset[f"mes_objetivo_{horizonte}m"].astype(str)
    return dataset.drop(columns=["mes_periodo"])


def resumir_aptitud_series(dataset: pd.DataFrame, horizonte: int) -> pd.DataFrame:
    objetivo = f"tasa_objetivo_{horizonte}m"
    trabajo = dataset.copy()
    trabajo["mes_periodo"] = pd.PeriodIndex(trabajo["mes"], freq="M")

    resumen = (
        trabajo.groupby(["banco", "rango_orden", "rango_monto"], dropna=False)
        .agg(
            primer_mes=("mes_periodo", "min"),
            ultimo_mes=("mes_periodo", "max"),
            meses_observados=("mes", "nunique"),
            observaciones_modelables=(objetivo, lambda serie: int(serie.notna().sum())),
            creditos_total=("total_creditos", "sum"),
            creditos_mediana=("total_creditos", "median"),
            tasa_minima=("tasa_ponderada", "min"),
            tasa_maxima=("tasa_ponderada", "max"),
            tasa_std=("tasa_ponderada", "std"),
        )
        .reset_index()
    )

    resumen["meses_esperados"] = resumen.apply(
        lambda fila: (fila["ultimo_mes"] - fila["primer_mes"]).n + 1,
        axis=1,
    )
    resumen["cobertura_meses"] = resumen["meses_observados"] / resumen["meses_esperados"]
    resumen["apta_modelo"] = (
        (resumen["meses_observados"] >= MIN_MESES_SERIE_MODELO)
        & (resumen["observaciones_modelables"] >= MIN_OBSERVACIONES_TARGET)
        & (resumen["creditos_total"] >= MIN_CREDITOS_SERIE)
        & (resumen["cobertura_meses"] >= 0.8)
    )

    resumen["primer_mes"] = resumen["primer_mes"].astype(str)
    resumen["ultimo_mes"] = resumen["ultimo_mes"].astype(str)
    return resumen.sort_values(
        ["apta_modelo", "observaciones_modelables", "creditos_total"],
        ascending=[False, False, False],
    ).reset_index(drop=True)


dataset_modelo_banco_rango = preparar_dataset_modelo_banco_rango(
    ponderada_banco_rango_mes,
    HORIZONTE_PREDICCION_MESES,
)
resumen_aptitud_banco_rango = resumir_aptitud_series(
    dataset_modelo_banco_rango,
    HORIZONTE_PREDICCION_MESES,
)
series_no_aptas_modelo = resumen_aptitud_banco_rango[~resumen_aptitud_banco_rango["apta_modelo"]].copy()

print("Dataset modelo banco-rango:", dataset_modelo_banco_rango.shape)
print("Series banco-rango totales:", len(resumen_aptitud_banco_rango))
print("Series aptas para modelo:", int(resumen_aptitud_banco_rango["apta_modelo"].sum()))
print("Series no aptas:", len(series_no_aptas_modelo))

display(resumen_aptitud_banco_rango.head(30))

if not series_no_aptas_modelo.empty:
    print("Series no aptas con mayor volumen de créditos:")
    display(series_no_aptas_modelo.sort_values("creditos_total", ascending=False).head(30))

Dataset modelo banco-rango: (3591, 13)
Series banco-rango totales: 143
Series aptas para modelo: 98
Series no aptas: 45


,banco,rango_orden,rango_monto,primer_mes,ultimo_mes,meses_observados,observaciones_modelables,creditos_total,creditos_mediana,tasa_minima,tasa_maxima,tasa_std,meses_esperados,cobertura_meses,apta_modelo
0,Bancolombia,1,Menor o igual a 3 SMLMV,2023-09,2026-03,31,29,641685,13553.0,21.948839,36.685589,4.572850,31,1.0,True
1,Banco Davibank,6,Mayor a 100 SMLMV,2023-09,2026-03,31,29,574174,6168.0,20.267688,41.828934,5.907003,31,1.0,True
2,Banco de Bogotá,1,Menor o igual a 3 SMLMV,2023-09,2026-03,31,29,398373,12404.0,23.879415,37.255322,4.460378,31,1.0,True
3,Banco de Bogotá,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,2023-09,2026-03,31,29,300010,9757.0,23.607385,36.538372,4.329408,31,1.0,True
4,Bancolombia,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,2023-09,2026-03,31,29,246880,8606.0,22.109839,32.950226,3.473849,31,1.0,True
5,Banco de Bogotá,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,2023-09,2026-03,31,29,219346,6894.0,23.142858,35.332474,4.088111,31,1.0,True
6,Bancolombia,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,2023-09,2026-03,31,29,208780,6239.0,20.929514,31.455167,3.460970,31,1.0,True
7,Banco Davivienda,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,2023-09,2026-03,31,29,187464,5557.0,21.112794,32.476997,3.681897,31,1.0,True
8,Banco Falabella,1,Menor o igual a 3 SMLMV,2023-09,2026-03,31,29,186107,6145.0,23.070540,37.774047,4.703002,31,1.0,True
9,Bancolombia,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,2023-09,2026-03,31,29,176153,5454.0,19.360994,30.524269,3.538235,31,1.0,True


Series no aptas con mayor volumen de créditos:


,banco,rango_orden,rango_monto,primer_mes,ultimo_mes,meses_observados,observaciones_modelables,creditos_total,creditos_mediana,tasa_minima,tasa_maxima,tasa_std,meses_esperados,cobertura_meses,apta_modelo
116,Bancien,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,2023-09,2024-08,12,10,2681,245.0,29.153673,39.800000,3.802063,12,1.000000,False
127,Banco Contactar,1,Menor o igual a 3 SMLMV,2024-03,2025-12,8,6,1876,193.5,14.000000,33.300000,6.352424,22,0.363636,False
114,Bancien,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,2023-09,2024-09,13,11,1603,125.0,14.030000,39.800000,6.725657,13,1.000000,False
130,Banco Contactar,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,2024-03,2025-12,7,5,1135,166.0,24.980000,33.300000,2.850727,22,0.318182,False
117,Bancien,1,Menor o igual a 3 SMLMV,2023-09,2024-08,12,10,314,25.5,29.154615,39.800000,3.811300,12,1.000000,False
124,Banco Pichincha,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,2023-10,2024-11,9,7,161,11.0,19.000000,29.486000,3.579412,14,0.642857,False
121,Banco Pichincha,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,2023-09,2025-05,10,8,118,4.5,18.298947,36.023333,5.484024,21,0.476190,False
131,Banco Contactar,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,2024-03,2025-12,7,5,109,14.0,24.980000,33.300000,2.850191,22,0.318182,False
133,Bancien,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,2023-09,2023-12,4,2,105,30.0,37.560000,39.800000,0.968843,4,1.000000,False
108,Banco Popular,1,Menor o igual a 3 SMLMV,2023-09,2026-02,20,18,101,2.0,24.090000,34.670000,4.042719,30,0.666667,False


In [36]:
continuidad_bancos = (
    raw_limpia.groupby(["banco", "banco_original"], dropna=False)
    .agg(
        primer_mes=("mes", "min"),
        ultimo_mes=("mes", "max"),
        registros=("mes", "size"),
        creditos=("numero_de_creditos", "sum"),
    )
    .reset_index()
    .sort_values(["banco", "primer_mes", "banco_original"])
)

bancos_con_multiples_nombres = (
    continuidad_bancos.groupby("banco")
    .filter(lambda grupo: grupo["banco_original"].nunique() > 1)
    .reset_index(drop=True)
)

print("Bancos con más de un nombre original asociado:")
display(bancos_con_multiples_nombres)

print("Continuidad específica Davibank / Scotiabank:")
display(
    continuidad_bancos[
        continuidad_bancos["banco"].str.contains("Davibank", case=False, na=False)
        | continuidad_bancos["banco_original"].str.contains("Scotiabank|Davibank", case=False, na=False)
    ]
)

Bancos con más de un nombre original asociado:


,banco,banco_original,primer_mes,ultimo_mes,registros,creditos
0,Banco Davibank,Scotiabank Colpatria S.A.,2023-09,2025-09,82071,738047
1,Banco Davibank,Banco Davibank,2025-10,2026-03,19223,59103


Continuidad específica Davibank / Scotiabank:


,banco,banco_original,primer_mes,ultimo_mes,registros,creditos
7,Banco Davibank,Scotiabank Colpatria S.A.,2023-09,2025-09,82071,738047
6,Banco Davibank,Banco Davibank,2025-10,2026-03,19223,59103


## 9. Exportación de archivos

Todos los archivos se guardan en la carpeta `outputs/` para no mezclar datos generados con notebooks. Los CSV se exportan con `utf-8-sig` para que Excel en Windows lea correctamente tildes y caracteres especiales.

In [39]:
EXPORTACIONES = {
    "tasas_raw_historico_completo.csv": raw_limpia_completa,
    "tasas_curadas_sin_rango_na.csv": raw_limpia,
    "registros_excluidos_rango_na.csv": rangos_excluidos_modelado,
    "tasas_ponderadas_banco_rango_mes.csv": ponderada_banco_rango_mes,
    "tasas_ponderadas_banco_mes.csv": ponderada_banco_mes,
    "tasas_ponderadas_total_mes.csv": ponderada_total_mes,
    "rangos_originales_api.csv": listado_rangos_originales,
    "plantilla_normalizacion_rangos.csv": plantilla_normalizacion_rangos,
    "diccionario_bancos.csv": diccionario_bancos,
    "diccionario_rangos.csv": diccionario_rangos,
    "validaciones_etl.csv": resumen_validaciones,
    "cobertura_banco.csv": cobertura_banco,
    "cobertura_mensual.csv": cobertura_mensual,
    "dataset_modelo_banco_rango_t2.csv": dataset_modelo_banco_rango,
    "resumen_aptitud_banco_rango.csv": resumen_aptitud_banco_rango,
    "series_no_aptas_modelo.csv": series_no_aptas_modelo,
    "continuidad_bancos.csv": continuidad_bancos,
    "bancos_con_multiples_nombres.csv": bancos_con_multiples_nombres,
}

for nombre_archivo, dataframe in EXPORTACIONES.items():
    ruta = OUTPUT_DIR / nombre_archivo
    dataframe.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Guardado: {ruta} ({len(dataframe):,} filas)")

Guardado: outputs\tasas_raw_historico_completo.csv (2,235,315 filas)
Guardado: outputs\tasas_curadas_sin_rango_na.csv (2,222,382 filas)
Guardado: outputs\registros_excluidos_rango_na.csv (12,933 filas)
Guardado: outputs\tasas_ponderadas_banco_rango_mes.csv (3,591 filas)
Guardado: outputs\tasas_ponderadas_banco_mes.csv (696 filas)
Guardado: outputs\tasas_ponderadas_total_mes.csv (31 filas)
Guardado: outputs\rangos_originales_api.csv (11 filas)
Guardado: outputs\plantilla_normalizacion_rangos.csv (11 filas)
Guardado: outputs\diccionario_bancos.csv (26 filas)
Guardado: outputs\diccionario_rangos.csv (10 filas)
Guardado: outputs\validaciones_etl.csv (14 filas)
Guardado: outputs\cobertura_banco.csv (25 filas)
Guardado: outputs\cobertura_mensual.csv (31 filas)
Guardado: outputs\dataset_modelo_banco_rango_t2.csv (3,591 filas)
Guardado: outputs\resumen_aptitud_banco_rango.csv (143 filas)
Guardado: outputs\series_no_aptas_modelo.csv (45 filas)
Guardado: outputs\continuidad_bancos.csv (26 filas)

In [41]:
# Exportación opcional a Excel con varias hojas.
# Si falta openpyxl, los CSV anteriores siguen siendo la salida principal del ETL.
ruta_excel = OUTPUT_DIR / "tasas_ponderadas_historico.xlsx"
try:
    with pd.ExcelWriter(ruta_excel) as writer:
        raw_limpia_completa.head(100000).to_excel(writer, sheet_name="raw_completa_muestra", index=False)
        raw_limpia.head(100000).to_excel(writer, sheet_name="curada_muestra", index=False)
        rangos_excluidos_modelado.head(100000).to_excel(writer, sheet_name="rangos_excluidos", index=False)
        ponderada_banco_rango_mes.to_excel(writer, sheet_name="banco_rango_mes", index=False)
        ponderada_banco_mes.to_excel(writer, sheet_name="banco_mes", index=False)
        ponderada_total_mes.to_excel(writer, sheet_name="total_mes", index=False)
        listado_rangos_originales.to_excel(writer, sheet_name="rangos_originales", index=False)
        plantilla_normalizacion_rangos.to_excel(writer, sheet_name="plantilla_rangos", index=False)
        diccionario_bancos.to_excel(writer, sheet_name="dicc_bancos", index=False)
        diccionario_rangos.to_excel(writer, sheet_name="dicc_rangos", index=False)
        resumen_validaciones.to_excel(writer, sheet_name="validaciones", index=False)
        dataset_modelo_banco_rango.to_excel(writer, sheet_name="modelo_banco_rango_t2", index=False)
        resumen_aptitud_banco_rango.to_excel(writer, sheet_name="aptitud_series", index=False)
        continuidad_bancos.to_excel(writer, sheet_name="continuidad_bancos", index=False)
    print(f"Excel guardado: {ruta_excel}")
except Exception as exc:
    print(f"No se generó Excel opcional: {exc}")

Excel guardado: outputs\tasas_ponderadas_historico.xlsx


## 10. Revisión pendiente

La interpretación narrativa de cada sección está en **[docs/INFORME_ETL.md](docs/INFORME_ETL.md)** y el marco general del proyecto en **[README.md](README.md)**.

Después de ejecutar el ETL completo, revisar especialmente:

- `diccionario_bancos.csv`: confirmar si los nombres canónicos son correctos o si hay bancos que deban agruparse/excluirse.
- `continuidad_bancos.csv`: validar cambios de nombre, especialmente `Scotiabank Colpatria` unificado como `Banco Davibank`.
- `diccionario_rangos.csv`: confirmar que todos los rangos históricos quedaron reconocidos y agrupados en las seis categorías SMLMV definidas.
- `registros_excluidos_rango_na.csv`: auditar los registros descartados por `N/A` o rango no reconocido antes de modelar.
- `resumen_aptitud_banco_rango.csv`: decidir si el modelado por banco-rango tiene suficiente historia o si conviene subir a banco-mes o total-mes.
- `dataset_modelo_banco_rango_t2.csv`: base candidata para entrenar modelos con objetivo a dos meses.
- `validaciones_etl.csv`: revisar tasas extremas, créditos no positivos, nulos y rangos no reconocidos.

Si se decide ampliar el alcance a todos los productos de consumo, solo se debe modificar `PARAMETROS["producto_de_credito"]` o quitar ese filtro en `construir_where_base`.